In [119]:
def get_stock_data(ticker):
    import yfinance as yf
    import numpy as np
    import pandas as pd
    stock = yf.download(ticker,start="2020-01-01", end="2025-01-01")
    data= pd.concat([stock, yf.download(ticker, start="2020-01-01", end="2025-01-01")['Volume']], axis=1)
    index = yf.download('^GSPC', start="2020-01-01", end="2025-01-01")

    # systematic risk factors
    data['Market_Returns'] = index['Close'].pct_change()
    for stock in ticker:
    # Use tuple to create MultiIndex column
        data[('Stock_Returns', stock)] = data[('Close', stock)].pct_change()
    
        rolling_window = 126
    
        data[('rolling_beta', stock)] = (
        data[('Stock_Returns', stock)].rolling(window=rolling_window).cov(data['Market_Returns']) /
        data['Market_Returns'].rolling(window=rolling_window).var()

    )
        
    return data

ticker = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA","NVDA"]
data = get_stock_data(ticker)
print(data.tail(30))

/var/folders/94/z2_z3thn10x6hzz91dytslr00000gn/T/ipykernel_28273/1337104420.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock = yf.download(ticker,start="2020-01-01", end="2025-01-01")
[*********************100%***********************]  6 of 6 completed
/var/folders/94/z2_z3thn10x6hzz91dytslr00000gn/T/ipykernel_28273/1337104420.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data= pd.concat([stock, yf.download(ticker, start="2020-01-01", end="2025-01-01")['Volume']], axis=1)
[*********************100%***********************]  6 of 6 completed
/var/folders/94/z2_z3thn10x6hzz91dytslr00000gn/T/ipykernel_28273/1337104420.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  index = yf.download('^GSPC', start="2020-01-01", end="2025-01-01")
[*********************100%***********************]  1 of 1 completed

            (Close, AAPL)  (Close, AMZN)  (Close, GOOGL)  (Close, MSFT)  \
Date                                                                      
2024-11-18     226.993362     201.699997      174.416245     411.891022   
2024-11-19     227.252197     204.610001      177.222015     413.902161   
2024-11-20     227.968948     202.880005      175.092804     411.623535   
2024-11-21     227.491104     198.380005      166.784912     409.846649   
2024-11-22     228.835037     197.119995      163.929352     413.946381   
2024-11-25     231.821518     201.449997      166.804794     415.723297   
2024-11-26     234.001663     207.860001      168.267380     424.855927   
2024-11-27     233.872253     205.740005      168.376831     419.892517   
2024-11-29     236.261444     207.889999      168.098236     420.359100   
2024-12-02     238.511261     210.710007      170.625443     427.824036   
2024-12-03     241.557495     213.440002      170.476181     428.042419   
2024-12-04     241.915863

In [108]:
index = yf.download('^GSPC', start="2020-01-01", end="2025-01-01")


/var/folders/94/z2_z3thn10x6hzz91dytslr00000gn/T/ipykernel_28273/1536891084.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  index = yf.download('^GSPC', start="2020-01-01", end="2025-01-01")
[*********************100%***********************]  1 of 1 completed


In [120]:
def data_preperation(data):
    import pandas as pd
    data = data.dropna()
    data = data.reset_index()
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.set_index('Date')
    return data
data = data_preperation(data)
print(data.tail(30))

            (Close, AAPL)  (Close, AMZN)  (Close, GOOGL)  (Close, MSFT)  \
Date                                                                      
2024-11-18     226.993362     201.699997      174.416245     411.891022   
2024-11-19     227.252197     204.610001      177.222015     413.902161   
2024-11-20     227.968948     202.880005      175.092804     411.623535   
2024-11-21     227.491104     198.380005      166.784912     409.846649   
2024-11-22     228.835037     197.119995      163.929352     413.946381   
2024-11-25     231.821518     201.449997      166.804794     415.723297   
2024-11-26     234.001663     207.860001      168.267380     424.855927   
2024-11-27     233.872253     205.740005      168.376831     419.892517   
2024-11-29     236.261444     207.889999      168.098236     420.359100   
2024-12-02     238.511261     210.710007      170.625443     427.824036   
2024-12-03     241.557495     213.440002      170.476181     428.042419   
2024-12-04     241.915863

In [121]:
def data_ranking(data, ticker):
    import pandas as pd
    ranking_df = pd.DataFrame()
    window = 126
    data["Cumulative_Market_Returns"] = index['Close'].pct_change(periods=126)

    for stock in ticker:
        data[('Cumulative_Stock_Returns', stock)] = data[('Close', stock)].pct_change(periods=126)
        data[('Standard_Deviation', stock)] = data[('Cumulative_Stock_Returns', stock)].std()
        data[('Z_Score', stock)] = (data['Cumulative_Market_Returns'] - data[('Cumulative_Stock_Returns', stock)]) / data[('Standard_Deviation', stock)]

        ranking_df = pd.concat([ranking_df, data[[('Z_Score', stock)]]], axis=1)
    ranking_df= ranking_df.sort_values(by = ticker)
        
    
    
    return ranking_df
ticker = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA","NVDA"]
data = data_ranking(data, ticker)
print(data)

KeyError: 'AAPL'

In [68]:
data['Mean_Retruns', "AAPL"] = data[('Stock_Returns', "AAPL")].rolling(window=126).mean()
print(data['Mean_Retruns', "AAPL"].tail(30))

Date
2024-11-18    0.001535
2024-11-19    0.001490
2024-11-20    0.001575
2024-11-21    0.001725
2024-11-22    0.001641
2024-11-25    0.001744
2024-11-26    0.001806
2024-11-27    0.001760
2024-11-29    0.001801
2024-12-02    0.001803
2024-12-03    0.001891
2024-12-04    0.001841
2024-12-05    0.001898
2024-12-06    0.001793
2024-12-09    0.002073
2024-12-10    0.001529
2024-12-11    0.001262
2024-12-12    0.001265
2024-12-13    0.001336
2024-12-16    0.001273
2024-12-17    0.001437
2024-12-18    0.001438
2024-12-19    0.001576
2024-12-20    0.001701
2024-12-23    0.001690
2024-12-24    0.001622
2024-12-26    0.001615
2024-12-27    0.001639
2024-12-30    0.001303
2024-12-31    0.001118
Name: (Mean_Retruns, AAPL), dtype: float64


In [80]:
print(data.tail(30))

            (Close, AAPL)  (Close, AMZN)  (Close, GOOGL)  (Close, MSFT)  \
Date                                                                      
2024-11-18     226.993362     201.699997      174.416245     411.891022   
2024-11-19     227.252197     204.610001      177.222015     413.902161   
2024-11-20     227.968948     202.880005      175.092804     411.623535   
2024-11-21     227.491104     198.380005      166.784912     409.846649   
2024-11-22     228.835037     197.119995      163.929352     413.946381   
2024-11-25     231.821518     201.449997      166.804794     415.723297   
2024-11-26     234.001663     207.860001      168.267380     424.855927   
2024-11-27     233.872253     205.740005      168.376831     419.892517   
2024-11-29     236.261444     207.889999      168.098236     420.359100   
2024-12-02     238.511261     210.710007      170.625443     427.824036   
2024-12-03     241.557495     213.440002      170.476181     428.042419   
2024-12-04     241.915863

In [88]:
print(data.columns)



Index([('Z_Score', 'NVDA')], dtype='object')


In [ ]:
def get_stock_data(ticker):
    import yfinance as yf
    import numpy as np
    import pandas as pd
    #download data
    stock = yf.download(ticker,start="2020-01-01", end="2025-01-01")
    data= pd.concat([stock, yf.download(ticker, start="2020-01-01", end="2025-01-01")['Volume']], axis=1)
    index = yf.download('^GSPC', start="2020-01-01", end="2025-01-01")

    # systematic risk factor Beta

    rolling_window = 126
    data['Market_Returns'] = index['Close'].pct_change()
    for stock in ticker:
        data[('Stock_Returns', stock)] = data[('Close', stock)].pct_change()
        data[('rolling_beta', stock)] = (
        data[('Stock_Returns', stock)].rolling(window=rolling_window).cov(data['Market_Returns']) /
        data['Market_Returns'].rolling(window=rolling_window).var()

    )
    
    # clean data 
    
    data = data.dropna()
    data = data.reset_index()
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.set_index('Date')

    # ranking data
    data["Cumulative_Market_Returns"] = data["Market_Returns"].pct_change(periods=rolling_window)
    for stock in ticker:
        data["Cumulative_Stock_Returns"] = data[('Stock_Returns', stock)].pct_change(periods=rolling_window)
        data[('Standard_Deviation', stock)] = data[('Stock_Returns', stock)].rolling(window=rolling_window).std()
        data[('Z_Score', stock)] = (data['Cumulative_Market_Returns'] - data[('Cumulative_Stock_Returns', stock)]) / data[('Standard_Deviation', stock)]
    